[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_03/13_ondas_estacionarias_y_adaptacion.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 13 — Ondas estacionarias y adaptación de impedancias

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 3**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Calcular la razón de onda estacionaria (ROE) y decir qué tan mal adaptada
   está una línea.
2. Hacer el balance de potencia: cuánta llega, cuánta vuelve y cuánta se
   entrega.
3. Diseñar un transformador de cuarto de onda para adaptar dos impedancias
   reales.
4. Explicar por qué esa adaptación solo funciona bien cerca de una frecuencia.

In [ ]:
# Preparación del entorno: funciona igual en Google Colab y en una copia local.
import sys
import urllib.request
from pathlib import Path

MODULOS = ["utilidades_notebook.py", "constantes_fisicas.py", "lineas_transmision.py"]
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)

# Se exige que estén *todos* los módulos, no solo la carpeta: así, si otro
# notebook ya creó `src/` en esta misma sesión, igual se descarga lo que falte.
raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if all((p / "src" / modulo).exists() for modulo in MODULOS)),
    None,
)
if raiz is None:  # Google Colab: descargar los módulos del curso.
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo in MODULOS:
        if not (raiz / "src" / modulo).exists():
            urllib.request.urlretrieve(URL_SRC + modulo, raiz / "src" / modulo)
sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from constantes_fisicas import VELOCIDAD_LUZ
from lineas_transmision import (
    coeficiente_reflexion,
    razon_onda_estacionaria,
    impedancia_entrada,
    impedancia_cuarto_de_onda,
    longitud_electrica,
    potencia_incidente,
    voltaje_en_la_linea,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 Por qué aparece un patrón fijo

En la línea viajan dos ondas: una hacia la carga y otra de vuelta. Al
sumarse, en algunos puntos coinciden en fase y se refuerzan; en otros llegan
en contrafase y se cancelan.

El resultado es un patrón de máximos y mínimos que **no se mueve**. Por eso
se llama onda estacionaria: la envolvente se queda quieta aunque las dos
ondas que la forman sí viajen.

### 2.2 La ROE en una frase

La ROE es el cociente entre el máximo y el mínimo de esa envolvente:

- ROE = 1: no hay reflexión, la línea está perfectamente adaptada.
- ROE = 2: reflexión moderada, típica en la práctica.
- ROE $\to\infty$: reflexión total, no se entrega nada de potencia.

Es la medida que aparece en cualquier instrumento de radiofrecuencia.

### 2.3 El truco del cuarto de onda

Una línea de un cuarto de longitud de onda **invierte** impedancias:
$Z_{\text{in}} = Z_t^2/Z_L$. Si usted elige $Z_t = \sqrt{Z_S Z_L}$, entonces
la entrada se ve exactamente como $Z_S$ y la adaptación es perfecta.

El precio es que la longitud de un cuarto de onda solo es correcta a una
frecuencia. Fuera de ella la adaptación se degrada, y eso limita el ancho de
banda.

## 3. Ecuaciones

**Reflexión y ROE:**

$$
\Gamma = \frac{Z_L - Z_0}{Z_L + Z_0},
\qquad
\mathrm{ROE} = \frac{1 + |\Gamma|}{1 - |\Gamma|}.
$$

**Voltaje a lo largo de la línea**, con $u$ la distancia a la carga medida en
longitudes de onda:

$$
\tilde{V}(u) = V^{+}\left(e^{-j2\pi u} + \Gamma\,e^{+j2\pi u}\right).
$$

**Balance de potencia:**

$$
P_i = \frac{|V^{+}|^{2}}{Z_0},
\qquad
P_r = |\Gamma|^{2}P_i,
\qquad
P_L = P_i - P_r = \left(1 - |\Gamma|^{2}\right)P_i .
$$

**Transformador de cuarto de onda** entre $Z_S$ y $Z_L$ reales:

$$
Z_t = \sqrt{Z_S Z_L},
\qquad
l = \frac{\lambda}{4} = \frac{u_p}{4f_0},
\qquad
u_p = \frac{c}{\sqrt{\varepsilon_r}} .
$$

**Casos particulares útiles**, para una línea de $\lambda/8$:

$$
Z_{\text{in}}^{\text{abierto}} = -jZ_0,
\qquad
Z_{\text{in}}^{\text{corto}} = +jZ_0 .
$$

Una línea corta se comporta como un condensador o como una bobina según cómo
esté terminada. Ésa es la base de los circuitos de microondas.

## 4. Qué significa físicamente

**Los mínimos no llegan a cero salvo con reflexión total.** La envolvente baja
hasta $|V^{+}|(1 - |\Gamma|)$. Si midiéndola usted encuentra ceros perfectos,
la carga está reflejando todo.

**Máximos y mínimos se alternan cada cuarto de longitud de onda.** El patrón
completo tiene período $\lambda/2$, con un máximo y un mínimo dentro de cada
período.

**La potencia reflejada va con $|\Gamma|^2$, no con $|\Gamma|$.** Con
$|\Gamma| = 0.45$ vuelve alrededor de un 20 % de la potencia. Un
$|\Gamma|$ que parece moderado puede significar una pérdida importante.

**El transformador de cuarto de onda es de banda estrecha.** A la frecuencia
de diseño la adaptación es exacta; al alejarse, la longitud deja de ser un
cuarto de onda y la reflexión reaparece. El gráfico de la sección 8 muestra
justamente esa "V" característica.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: onda estacionaria ---
Z0 = 50.0                # impedancia característica [ohm]
ZL = 25.0 + 25.0j        # impedancia de la carga [ohm]
voltaje_incidente = 10.0  # amplitud RMS de la onda incidente [V]

# --- Problema 2: transformador de cuarto de onda ---
Z_fuente = 50.0     # impedancia de la fuente [ohm]
Z_carga = 100.0     # impedancia de la carga a adaptar [ohm]
eps_r = 2.25        # permitividad relativa del sustrato
frecuencia_diseno = 1.0e9  # frecuencia de diseño f_0 [Hz]

## 6. Implementación

### 6.1 Problema 1 — reflexión, ROE y potencias

In [ ]:
Gamma = coeficiente_reflexion(ZL, Z0)
roe = razon_onda_estacionaria(Gamma)

P_incidente = potencia_incidente(voltaje_incidente, Z0)
P_reflejada = abs(Gamma) ** 2 * P_incidente
P_entregada = P_incidente - P_reflejada

### 6.2 Problema 2 — diseño del transformador

In [ ]:
Z_transformador = impedancia_cuarto_de_onda(Z_fuente, Z_carga)

velocidad = VELOCIDAD_LUZ / np.sqrt(eps_r)
longitud_fisica = velocidad / (4.0 * frecuencia_diseno)

# A la frecuencia de diseño, la línea de lambda/4 invierte la impedancia.
Zin_en_diseno = Z_transformador**2 / Z_carga

# Línea de lambda/8 terminada en abierto y en corto.
Zin_abierto_lambda_octavo = -1.0j * Z0
Zin_corto_lambda_octavo = 1.0j * Z0

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Reflexión, parte real", "Re(Gamma)", Gamma.real, "-"),
        ("Reflexión, parte imaginaria", "Im(Gamma)", Gamma.imag, "-"),
        ("Módulo de la reflexión", "|Gamma|", abs(Gamma), "-"),
        ("Fase de la reflexión", "arg(Gamma)", np.rad2deg(np.angle(Gamma)), "grados"),
        ("Razón de onda estacionaria", "ROE", roe, "-"),
        ("Potencia incidente", "P_i", P_incidente, "W"),
        ("Potencia reflejada", "P_r", P_reflejada, "W"),
        ("Potencia entregada a la carga", "P_L", P_entregada, "W"),
        ("Fracción entregada", "P_L/P_i", P_entregada / P_incidente, "-"),
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Impedancia del transformador", "Z_t", Z_transformador, "ohm"),
        ("Longitud física del tramo", "l", longitud_fisica, "m"),
        ("Longitud física del tramo", "l", longitud_fisica * 1000.0, "mm"),
        ("Impedancia vista en f_0", "Z_in", Zin_en_diseno, "ohm"),
        ("Línea abierta de lambda/8", "Im(Z_in)", Zin_abierto_lambda_octavo.imag, "ohm"),
        ("Línea en corto de lambda/8", "Im(Z_in)", Zin_corto_lambda_octavo.imag, "ohm"),
    ]
)

In [ ]:
print(f"Z_in en f_0 = {Zin_en_diseno:.6f} ohm")
print(f"Z_fuente    = {Z_fuente:.6f} ohm")
assert np.isclose(Zin_en_diseno, Z_fuente), "El transformador no adapta"
print("La adaptación en la frecuencia de diseño es exacta.")

## 8. Visualización

A la izquierda, el patrón de onda estacionaria. A la derecha, cuánto se
degrada la adaptación al alejarse de la frecuencia de diseño.

In [ ]:
fig, (eje_patron, eje_banda) = plt.subplots(1, 2, figsize=(9.5, 4.0))

u = np.linspace(0.0, 1.0, 700)
voltaje = voltaje_en_la_linea(voltaje_incidente, Gamma, u)
maximo = voltaje_incidente * (1.0 + abs(Gamma))
minimo = voltaje_incidente * (1.0 - abs(Gamma))

eje_patron.plot(u, abs(voltaje))
eje_patron.axhline(maximo, color="black", linestyle=":", label="máximo")
eje_patron.axhline(minimo, color="gray", linestyle=":", label="mínimo")
eje_patron.set_xlabel("Distancia a la carga, en longitudes de onda")
eje_patron.set_ylabel("|V| RMS (V)")
eje_patron.set_title(f"Onda estacionaria (ROE = {roe:.2f})")
eje_patron.legend(fontsize=8)

razon_frecuencias = np.linspace(0.5, 1.5, 600)
# A la frecuencia f, la longitud fija de lambda_0/4 equivale a beta*l = (pi/2)(f/f_0).
beta_l = 0.5 * np.pi * razon_frecuencias
Zin_banda = impedancia_entrada(Z_carga, Z_transformador, beta_l)
Gamma_banda = coeficiente_reflexion(Zin_banda, Z_fuente)

eje_banda.plot(razon_frecuencias, abs(Gamma_banda), color="tab:orange")
eje_banda.axvline(1.0, color="black", linestyle="--", label="f_0")
eje_banda.set_xlabel("f / f_0")
eje_banda.set_ylabel("|Gamma| a la entrada")
eje_banda.set_title("La adaptación solo vale cerca de f_0")
eje_banda.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**Una ROE de 2.6 significa perder un 20 % de la potencia.** El balance de la
tabla lo confirma: de 2 W incidentes vuelven 0.4 W y llegan 1.6 W a la carga.

**El patrón se repite dos veces en el gráfico.** El eje llega hasta una
longitud de onda completa, y se ven dos máximos y dos mínimos: el período es
$\lambda/2$, como anticipamos.

**El mínimo no toca el cero.** Queda en $10(1 - 0.447) = 5.5$ V. Solo con
reflexión total la envolvente llegaría a cero.

**El transformador adapta exactamente en $f_0$.** El `assert` verifica que
$Z_{\text{in}} = 50~\Omega$ justo en la frecuencia de diseño. En el gráfico
de la derecha, $|\Gamma|$ cae hasta cero en $f/f_0 = 1$ y sube a ambos lados
formando una "V".

**El tramo mide unos 5 cm.** Con $\varepsilon_r = 2.25$ a 1 GHz. Eso es
perfectamente fabricable en una placa de circuito impreso, y es exactamente
así como se hacen las adaptaciones en la práctica.

## 10. Ejercicios para experimentar

            1. Ponga `ZL = 50.0`. ¿Cuánto vale la ROE? ¿Cómo queda el patrón de la
               izquierda? ¿Cuánta potencia se entrega?
            2. Ponga `ZL = 0.0` (cortocircuito). ¿Cuánto vale la ROE? ¿Y la potencia
               entregada? ¿Tiene sentido físico?
            3. Ponga `ZL = 100.0` real. Compare la ROE con el caso `ZL = 25.0`: los dos
               dan lo mismo. ¿Por qué?
            4. Cambie `Z_carga` a `200.0`. ¿Cuánto vale ahora $Z_t$? ¿Es un valor
               fabricable en una placa común?
            5. Suba `frecuencia_diseno` a `5.0e9`. ¿Cuánto mide el tramo? ¿Sigue siendo
               práctico?
            6. Mire el gráfico de la derecha y estime en qué rango de frecuencias
               $|\Gamma| < 0.1$. Exprese ese ancho de banda como porcentaje de $f_0$.